# Prompt Chaining, itinerario paso a paso

Clasificación: **Workflow.** El orden vuelos→alojamiento→actividades lo fijo yo en el código; ningún agente decide llamar a otro.

Planificar un viaje tiene un orden natural: primero sabes cuándo vuelas, luego reservas alojamiento para esas fechas, luego encajas actividades en los días que quedan. Cada paso necesita la salida del anterior, así que prompt chaining encaja bien.

Los agentes y tareas van en YAML (`config/agents.yaml`, `config/tasks.yaml`), la orquestación en `viajes_crew.py` con `@CrewBase`.

In [1]:
!uv pip install -r requirements.txt --quiet

In [2]:
from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

## Por qué YAML externo en vez de definir todo en código

CrewAI recomienda configurar agentes y tareas en YAML como approach por defecto. Las razones son prácticas:

1. **Separar qué hace cada agente de cómo se orquesta.** Los prompts (role, goal, backstory, description) cambian mucho más que la lógica de conexión. Con YAML puedes ajustar un prompt sin tocar Python.

2. **Variables interpoladas.** Los YAML aceptan `{variable}` que se resuelven con los `inputs` del kickoff. Esto te permite reusar la misma crew para distintos destinos, presupuestos o duraciones sin duplicar código.

3. **El código Python queda mínimo.** La clase `@CrewBase` solo conecta agentes con tareas y define el proceso. Si miras `viajes_crew.py`, casi no hay lógica: los métodos `@agent` y `@task` solo devuelven objetos con la config que viene del YAML.

4. **Es lo que genera `crewai create`.** El CLI de CrewAI scaffoldea la estructura `config/agents.yaml` + `config/tasks.yaml` + `crew.py` por defecto, así que seguir esta convención hace que el proyecto sea familiar para cualquiera que haya usado el framework.

La alternativa (definir todo inline en Python) funciona, pero en cuanto tienes 3-4 agentes los bloques de texto en el código se vuelven difíciles de leer y de mantener.

## Configuración de agentes (`config/agents.yaml`)

Cada clave de primer nivel (p.ej. `vuelos:`) es el identificador del agente, referenciado desde `tasks.yaml` y desde `@CrewBase`.

| Atributo | Tipo | Qué hace |
|----------|------|----------|
| `role` | `str` | La función del agente en la crew. Determina qué tipo de tareas aborda. |
| `goal` | `str` | Objetivo que el agente intenta cumplir. Guía sus decisiones en cada iteración. |
| `backstory` | `str` | Contexto y personalidad. Le da al LLM un personaje consistente para razonar. |
| `llm` | `str` | Modelo a usar: `provider/model` o solo el nombre (p.ej. `gpt-4.1-mini`). |
| `verbose` | `bool` | Si `true`, imprime cada paso de razonamiento. Útil en desarrollo. |

In [3]:
%pycat config/agents.yaml

vuelos:
  role: "Especialista en Vuelos"
  goal: "Encontrar las mejores opciones de vuelo dentro de presupuesto"
  backstory: >
    Conoces tarifas y aerolineas para rutas europeas.
  llm: "gpt-4.1-mini"
  verbose: true

alojamiento:
  role: "Especialista en Alojamientos"
  goal: "Encontrar el alojamiento con mejor relacion precio-calidad comparando plataformas"
  backstory: >
    Comparas Airbnb y Booking para encontrar la opcion que mas rinda
    dentro del presupuesto restante.
  llm: "gpt-4.1-mini"
  verbose: true

actividades:
  role: "Especialista en Actividades"
  goal: "Proponer un plan de actividades dentro del presupuesto restante"
  backstory: >
    Conoces atracciones, restaurantes y experiencias locales.
  llm: "gpt-4.1-mini"
  verbose: true

coche:
  role: "Especialista en Rutas en Coche"
  goal: "Planificar rutas en coche entre las zonas del viaje, ya sea coche propio o de alquiler"
  backstory: >
    Calculas rutas por carretera, tiempos de conduccion, peajes y costes
 

## Configuración de tareas (`config/tasks.yaml`)

Cada clave de primer nivel (p.ej. `vuelos_task:`) es el identificador de la tarea, referenciado desde `@CrewBase`.

| Atributo | Tipo | Qué hace |
|----------|------|----------|
| `description` | `str` | Lo que el agente debe hacer. Acepta variables `{variable}` que se interpolan con los inputs del kickoff. |
| `expected_output` | `str` | Cómo se ve la respuesta terminada. El agente lo usa para saber cuándo parar. |
| `agent` | `str` | Agente asignado (debe coincidir con una clave en `agents.yaml`). |

In [4]:
%pycat config/tasks.yaml

vuelos_task:
  description: >
    Propon 2-3 opciones de vuelo a {destino} para {personas} personas y {dias} dias.
    Presupuesto total del viaje: {presupuesto} EUR.
  expected_output: >
    2-3 opciones de vuelo con precio aproximado y fechas.
  agent: vuelos

alojamiento_task:
  description: >
    Propon 2 opciones de alojamiento en {destino} para {personas} personas y {dias} noches.
    Compara una opcion en Airbnb y otra en Booking.
    Presupuesto total del viaje: {presupuesto} EUR.
  expected_output: >
    2 opciones de alojamiento (una de cada plataforma) con precio por noche y zona.
  agent: alojamiento

actividades_task:
  description: >
    Propon un plan de actividades para {dias} dias en {destino} para {personas} personas.
    Presupuesto total del viaje: {presupuesto} EUR.
  expected_output: >
    Actividades dia por dia con coste estimado.
  agent: actividades

transporte_task:
  description: >
    Propon opciones de transporte para moverse entre los puntos del viaje en 

## Cómo funciona la Crew (`viajes_crew_basic.py`)

Una Crew es el objeto que junta agentes, tareas y proceso de ejecución. La clase Python usa `@CrewBase` y un conjunto de decoradores que conectan los YAML con la lógica:

### Decoradores

| Decorador | Para qué sirve |
|-----------|----------------|
| `@CrewBase` | Marca la clase como crew. Carga automáticamente `agents_config` y `tasks_config` desde los YAML. |
| `@agent` | Registra un método como fábrica de agente. El nombre del método debe coincidir con la clave en `agents.yaml`. |
| `@task` | Registra un método como fábrica de tarea. El nombre del método debe coincidir con la clave en `tasks.yaml`. |
| `@crew` | Marca el método que construye y devuelve el objeto `Crew` final. |

### Atributos del `Crew()`

| Atributo | Qué hace |
|----------|----------|
| `agents` | Lista de agentes que participan. Se pueden recoger automáticamente con `self.agents`, o listarlos a mano como en este fichero. |
| `tasks` | Lista de tareas a ejecutar, en el orden en que se pasan. |
| `process` | Cómo se ejecutan las tareas. `Process.sequential` las corre una tras otra (la salida de cada una pasa como contexto a la siguiente). |
| `verbose` | Si `True`, imprime logs de ejecución de toda la crew. |

### Atributos usados en las tareas (dentro de Python)

| Atributo | Qué hace |
|----------|----------|
| `context` | Lista de tareas cuya salida se inyecta como contexto. En `itinerario_task` se usa para recibir los resultados de vuelos, alojamiento, actividades y transporte. |

In [5]:
%pycat viajes_crew_basic.py

from __future__ import annotations

from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task

@CrewBase
class ViajesCrewBasic:
    """Crew secuencial para plan de viaje sin tools ni mcps asignadas a los agentes."""

    agents_config = "./config/agents.yaml"
    tasks_config = "./config/tasks.yaml"

    @agent
    def vuelos(self) -> Agent:
        return Agent(config=self.agents_config["vuelos"])

    @agent
    def alojamiento(self) -> Agent:
        return Agent(config=self.agents_config["alojamiento"])

    @agent
    def actividades(self) -> Agent:
        return Agent(config=self.agents_config["actividades"])
    
    @agent
    def transporte(self) -> Agent:
        return Agent(config=self.agents_config["transporte"])

    @agent
    def coche(self) -> Agent:
        return Agent(config=self.agents_config["coche"])

    @agent
    def itinerario(self) -> Agent:
        return Agent(config=self.agents_config["itinerario"])

    @task
 

In [6]:
from viajes_crew_basic import ViajesCrewBasic

inputs = {
    "destino": "Islandia",
    "dias": 10,
    "personas": 2,
    "presupuesto": 2200,
}

trip = ViajesCrewBasic()
result = await trip.crew().kickoff_async(inputs=inputs)
print(result.raw)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: ViajesCrewBasic                                                                                          │
│  ID: e0f950bb-abc5-46f4-8da4-4ed73a7198e1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: vuelos_task                                                                                              │
│  ID: 92bddf20-76a4-4d72-b28a-c9877d088022                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Especialista en Vuelos                                                                                  │
│                                                                                                                 │
│  Task: Propon 2-3 opciones de vuelo a Islandia para 2 personas y 10 dias. Presupuesto total del viaje: 2200     │
│  EUR.                                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Especialista en Vuelos                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Para un viaje de 2 personas a Islandia por 10 días con un presupuesto total de 2200 EUR, considerando que      │
│  parte del presupuesto será destinado al alojamiento y gastos en destino, te propongo 3 opciones de vuelos con  │
│  diferentes niveles de flexibilidad y precios aproximados. Supongo que la salida será desde una ciudad europea  │
│  importante, como Madrid o Barcelona, y fechas indicativas en temporada media (septiembre-octubre 2024) para    │
│  mejores tarifas.                                                                                               │
│                                                                                                                 │
│  ### Opción 1: Vuelo directo con low cost (Icelandair / PLAY Airlines)                                          │
│  - **Vuelo:** Madrid o Barcelona - Reykjavik (aeropuerto KEF)                                                   │
│  - **Fechas:** Ida 5 de Octubre - Vuelta 15 de Octubre 2024                                                     │
│  - **Precio aprox por persona:** 180-220 EUR ida y vuelta (tarifa básica, equipaje de mano)                     │
│  - **Precio total para 2 personas:** 360-440 EUR                                                                │
│  - **Ventajas:** vuelo directo, buena frecuencia, equipaje adicional con costo extra                            │
│  - **Aerolineas:** Icelandair, PLAY                                                                             │
│                                                                                                                 │
│  ### Opción 2: Vuelo con escala corta (Lufthansa + SAS)                                                         │
│  - **Ruta:** Madrid - Frankfurt (Lufthansa) - Reykjavik (SAS)                                                   │
│  - **Fechas:** Ida 6 de Octubre - Vuelta 16 de Octubre 2024                                                     │
│  - **Precio aprox por persona:** 280-320 EUR ida y vuelta incluyendo 1 pieza de equipaje facturado              │
│  - **Precio total para 2 personas:** 560-640 EUR                                                                │
│  - **Ventajas:** equipaje incluido, posibilidad de acumular millas, más flexibilidad                            │
│  - **Duración:** 5-7 horas con escala de 1-3 horas                                                              │
│                                                                                                                 │
│  ### Opción 3: Vuelo con aerolínea low cost desde Londres (easyJet + Ryanair + vuelo separado a Londres) –      │
│  Opción para viajeros flexibles                                                                                 │
│  - **Ruta:** Madrid - Londres (easyJet/Ryanair), Londres - Reykjavik (easyJet/PLAY)                             │
│  - **Fechas:** Ida 4 de Octubre - Vuelta 14 de Octubre 2024                                                     │
│  - **Precio aprox total por persona:** 200-250 EUR ida y vuelta por ambos vuelos con baja tarifa (equipaje de   │
│  mano)                                                                                                          │
│  - **Precio total para 2 personas:** 400-500 EUR                                                                │
│  - **Nota:** Debe considerarse tiempo para conexiones y

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Director de Itinerario                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Itinerario final para 2 personas en Islandia (10 días) – Presupuesto máximo 2200 EUR                         │
│                                                                                                                 │
│  Basado en vuelos low cost directos (Opción 1), alojamiento guesthouse en Reykjavik con desayuno (Booking –     │
│  Guesthouse Odinn), alquiler de coche limitado y combinación de tours económicos y transporte público,          │
│  manteniendo el presupuesto total.                                                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Resumen presupuestario final estimado                                                                       │
│                                                                                                                 │
│  | Concepto                      | Coste EUR  |                                                                 │
│  |------------------------------|------------|                                                                  │
│  | Vuelos low cost (2 pax ida y vuelta) | 400        |                                                          │
│  | Alojamiento (guesthouse 10 noches)   | 1300       |                                                          │
│  | Transporte (Flybus + 2 días coche + tours + gasolina) | 400        |                                         │
│  | Actividades (entradas, tours, piscinas) | 250        |                                                       │
│  | Comidas (desayuno incluido, cenas y picnic) | 150        |                                                   │
│  | **TOTAL ESTIMADO**            | **2200 EUR** |                                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # Día 1 – Llegada y Reykjavik centro                                                                           │
│                                                                                                                 │
│  | Hora         | Actividad                                           |                                         │
│  |--------------|----------------------------------------------------|                                          │
│  | 15:00        | Llegada aeropuerto KEF                              |                                         │
│  | 15:30 – 16:15| Flybus aeropuerto → Guesthouse Odinn (45 min, 35 EUR p/p ida y vuelta, 70 EUR total) |        │
│  | 17:00 – 19:00| Check-in y paseo por Hallgrímskirkja, Harpa Concert Hall, y calle Laugavegur |                │
│  | 19:30 – 20:30| Cena económica (p.ej. Bæjarins Beztu Pylsur) ~20 EUR |                                        │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: itinerario_task                                                                                          │
│  Agent: Director de Itinerario                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

# Itinerario final para 2 personas en Islandia (10 días) – Presupuesto máximo 2200 EUR

Basado en vuelos low cost directos (Opción 1), alojamiento guesthouse en Reykjavik con desayuno (Booking – Guesthouse Odinn), alquiler de coche limitado y combinación de tours económicos y transporte público, manteniendo el presupuesto total.

---

## Resumen presupuestario final estimado

| Concepto                      | Coste EUR  |
|------------------------------|------------|
| Vuelos low cost (2 pax ida y vuelta) | 400        |
| Alojamiento (guesthouse 10 noches)   | 1300       |
| Transporte (Flybus + 2 días coche + tours + gasolina) | 400        |
| Actividades (entradas, tours, piscinas) | 250        |
| Comidas (desayuno incluido, cenas y picnic) | 150        |
| **TOTAL ESTIMADO**            | **2200 EUR** |

---

# Día 1 – Llegada y Reykjavik centro

| Hora         | Actividad                                           |
|--------------|---------------------------------------------------

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: ViajesCrewBasic                                                                                          │
│  ID: e0f950bb-abc5-46f4-8da4-4ed73a7198e1                                                                       │
│  Final Output: # Itinerario final para 2 personas en Islandia (10 días) – Presupuesto máximo 2200 EUR           │
│                                                                                                                 │
│  Basado en vuelos low cost directos (Opción 1), alojamiento guesthouse en Reykjavik con desayuno (Booking –     │
│  Guesthouse Odinn), alquiler de coche limitado y combinación de tours económicos y transporte público,          │
│  manteniendo el presupuesto total.                                                                              │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## Resumen presupuestario final estimado                                                                       │
│                                                                                                                 │
│  | Concepto                      | Coste EUR  |                                                                 │
│  |------------------------------|------------|                                                                  │
│  | Vuelos low cost (2 pax ida y vuelta) | 400        |                                                          │
│  | Alojamiento (guesthouse 10 noches)   | 1300       |                                                          │
│  | Transporte (Flybus + 2 días coche + tours + gasolina) | 400        |                                         │
│  | Actividades (entradas, tours, piscinas) | 250        |                                                       │
│  | Comidas (desayuno incluido, cenas y picnic) | 150        |                                                   │
│  | **TOTAL ESTIMADO**            | **2200 EUR** |                                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  # Día 1 – Llegada y Reykjavik centro                                                                           │
│                                                                                                                 │
│  | Hora         | Actividad                                           |                                         │
│  |--------------|----------------------------------------------------|                                          │
│  | 15:00        | Llegada aeropuerto KEF                              |                                         │
│  | 15:30 – 16:15| Flybus aeropuerto → Guesthouse Odinn (45 min, 35 EUR p/p ida y vuelta, 70 EUR total) |        │
│  | 17:00 – 19:00| Check-in y paseo por Hallgrímskirkja, Harpa Concert Hall, y calle Laugavegur |                │
│  | 19:30 – 20:30| Cena económica (p.ej. Bæjarins Beztu Pylsur) ~20 EUR |                                        │
│                                                       